In [1]:
import os
import sys
root_dir = "/Users/livardywufianto/Projects/PatchTST"
sys.path.append(
    os.path.join(
        root_dir, 
        "PatchTST_self_supervised"
    )
)

In [2]:
import numpy as np
import pandas as pd
import os
import torch
from torch import nn

from src.models.patchTST import PatchTST
from src.learner import Learner, transfer_weights
from src.callback.core import *
from src.callback.tracking import *
from src.callback.patch_mask import *
from src.callback.transforms import *
from src.metrics import *
from src.basics import set_device
from datautils import *

/Users/livardywufianto/.pyenv/versions/3.10.13/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
args = {
    # Pretraining and Finetuning
    'is_finetune': 0,
    'is_linear_probe': 0,
    
    # Dataset and dataloader
    'dset_finetune': 'etth1', # dataset name
    'context_points': 512, # sequence length
    'target_points': 96, # forecast horizon
    'batch_size': 64, # batch size
    'num_workers': 0,
    'scaler': 'standard',
    'features': 'M', # for multivariate or Univariate
    
    # Patching (Section 1 of the model)
    'patch_len': 12, 
    'stride': 12,
    
    # RevIN (Instance Norm)
    'revin': 1, # reversible instance normalization
    
    # Model Architecture (Sections 2 & 3)
    'n_layers': 3,
    'n_heads': 16,
    'd_model': 128,
    'd_ff': 256,
    'dropout': 0.2,
    'head_dropout': 0.2,
    
    # Optimization
    'n_epochs_finetune': 20,
    'lr': 1e-4,
    
    # Tracking and Paths
    'pretrained_model': None,
    'finetuned_model_id': 1,
    'model_type': 'based_model'
}

from argparse import Namespace
args = Namespace(**args)

In [4]:
from src.models.ct_patch_tst import CTPatchTST

def get_ct_patch_tst_model(c_in, args, head_type, weight_path=None):
    """
    c_in: number of variables
    """
    # get number of patches
    num_patch = (max(args.context_points, args.patch_len)-args.patch_len) // args.stride + 1    
    print('number of patches:', num_patch)
    
    # get model
    # model = PatchTST(c_in=c_in, 
    #             target_dim=args.target_points,
    #             patch_len=args.patch_len,
    #             stride=args.stride,
    #             num_patch=num_patch,
    #             n_layers=args.n_layers,
    #             n_heads=args.n_heads,
    #             d_model=args.d_model,
    #             shared_embedding=True,
    #             d_ff=args.d_ff,                        
    #             dropout=args.dropout,
    #             head_dropout=args.head_dropout,
    #             act='relu',
    #             head_type=head_type,
    #             res_attention=False
    # )    
    model_args = {
        'c_in': c_in,
        'target_dim': args.target_points,
        'patch_len': args.patch_len,
        'stride': args.stride,
        'num_patch': num_patch,
        'n_layers': args.n_layers,
        'n_heads': args.n_heads,
        'd_model': args.d_model,
        'shared_embedding': True,
        'd_ff': args.d_ff,
        'dropout': args.dropout,
        'head_dropout': args.head_dropout,
        'act': 'relu',
        'head_type': head_type,
        'res_attention': False
    }    

    for key, value in model_args.items():
        print(f"{key}: {value}")
    model = CTPatchTST(**model_args)
    
    if weight_path: model = transfer_weights(weight_path, model)
    # print out the model size
    print('number of model params', sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

In [5]:
args.dset = args.dset_finetune
dls = get_dls(args)

In [7]:
model = get_ct_patch_tst_model(dls.vars, args, head_type='prediction')

number of patches: 42
c_in: 7
target_dim: 96
patch_len: 12
stride: 12
num_patch: 42
n_layers: 3
n_heads: 16
d_model: 128
shared_embedding: True
d_ff: 256
dropout: 0.2
head_dropout: 0.2
act: relu
head_type: prediction
res_attention: False


AttributeError: cannot assign module before Module.__init__() call